In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path
from tqdm import tqdm
import pickle

# Generate hotspots of unfairness based on movement patterns

#### Read the Atlanta's 2020 US Census blocks and the stop segments

In [ ]:
path_atlanta_blocks = './experiments/atlanta_census_blocks.zip'
atlanta_blocks = gpd.read_file(path_atlanta_blocks)['geometry'].to_crs("EPSG:4326").to_frame()
atlanta_blocks


path_stop_df = './data_simulator/huge_dataset/dataset_simulator_trajectories.compressed.parquet.stops.parquet'
stop_df = pd.read_parquet(path_stop_df)
stop_df = gpd.GeoDataFrame(stop_df, 
                           geometry=gpd.points_from_xy(stop_df.lng, stop_df.lat), 
                           crs="EPSG:4326").loc[:, ['uid', 'geometry']]
display(stop_df)

#### Estimate a metric CRS we can use for some manipulations for objects that fall over the area covered by the census blocks.

In [ ]:
orig_crs = atlanta_blocks.crs
metric_crs = atlanta_blocks.estimate_utm_crs()
# print(orig_crs, metric_crs)


# Reproject both blocks and stop segments' centroids.
atlanta_blocks = atlanta_blocks.to_crs(metric_crs)
stop_df = stop_df.to_crs(metric_crs)

#### Filter out the blocks that do not contain any stop segment.

In [ ]:
# Associated each stop segment to a census block via its centroid.
# This will be useful when building hotspots made of multiple separate regions: we can use the candidate
# generation algorithm to see where there are objects associated with more than 1 separate region, and use them
# as seeds to build this kind of hotspots.
mapped_stops = stop_df.sjoin(atlanta_blocks,
                             how="left",
                             predicate="within")[['uid', 'index_right']]

# Filter out the blocks that do not contain any stop segment.
list_nonempty_blocks = mapped_stops['index_right'].unique()
# display(list_nonempty_blocks)

# Store the user IDs associated with each stop in a numpy array.
stop_uid_values = stop_df["uid"].to_numpy()

# Get the rtree associated with df_stops.
rtree_stops = stop_df.sindex

# Count the occurrences of each pair (uid, block), i.e., find out the number of stops each uid has in some block.
map_uid_blocks = mapped_stops.groupby(['uid', 'index_right']).size()
# display(map_uid_blocks)

sel_atlanta_blocks = atlanta_blocks.loc[list_nonempty_blocks].copy()
# display(sel_atlanta_blocks)
# sel_atlanta_blocks.plot()


# Remove some dataframes from memory (no more necessary for here on).
del atlanta_blocks, mapped_stops, stop_df

#### Generate multi-region hotspots of unfairness based on movement patterns

In [ ]:
from src.gen_unfair_multiregion_hotspots import gen_unfair_datasets_multiregion_hotspots
from joblib import Parallel, delayed
import itertools


total_datasets = 1000
num_objs_per_hotspot = 200
parallel = True
if parallel :
    n_jobs = -1
    batch_size = 10

    # split total_datasets into batches of size 'batch_size'
    batch_sizes = [batch_size] * (total_datasets // batch_size)
    remainder = total_datasets % batch_size
    if remainder: batch_sizes.append(remainder)

    # parallel execution
    datasets_batches = Parallel(n_jobs=n_jobs, 
                                backend="threading", 
                                verbose=50,
                                max_nbytes="10K", # very low on purpose, to force memmapping
                                mmap_mode="r")(delayed(gen_unfair_datasets_multiregion_hotspots)(
                                    sel_atlanta_blocks, 
                                    map_uid_blocks, stop_uid_values, rtree_stops,
                                    num_unfair_datasets=curr_batch_size, 
                                    num_hotspots_per_dataset=1, 
                                    num_regions_per_hotspot=2,
                                    num_objs_per_hotspot=num_objs_per_hotspot,
                                    global_pos_rate=0.6, hotspots_pos_rate=0.4)
        for curr_batch_size in batch_sizes
    )
    # Concatenate the datasets generated in the various chunks.
    datasets = list(itertools.chain(*datasets_batches))
    del datasets_batches


# Sequential execution.
else :
    datasets = gen_unfair_datasets_multiregion_hotspots(sel_atlanta_blocks, 
                                                        map_uid_blocks, stop_uid_values, rtree_stops,
                                                        num_unfair_datasets=total_datasets, 
                                                        num_hotspots_per_dataset=1, 
                                                        num_regions_per_hotspot=2,
                                                        num_objs_per_hotspot=num_objs_per_hotspot,
                                                        global_pos_rate=0.6, hotspots_pos_rate=0.4)

In [ ]:
# Find out the IDs of the objects that are associated with at least an hotspot.
list_num_objs = [np.unique(np.concatenate(h[1])).size for h in datasets]

# Compute the mean number of objects belonging to an hotspot.append
# Ideally, the mean should be equal to the target size of an hotspot times the number of hotspots per dataset.
print( np.mean(list_num_objs), np.std(list_num_objs), np.min(list_num_objs), np.max(list_num_objs) )

### Write the synthetic unfair labels to disk

In [ ]:
path_unfair_dataset = './experiments/unfair_datasets.pkl'
with open(path_unfair_dataset, "wb") as f:
    pickle.dump(datasets, f)

### DEBUG: Plot the original shape of a polygon, and its shrunken+rotated+translated version.

**DEBUG**: plot a simple Folium map of the Atlanta's tracts -- nonempty vs empty.